In [1]:
#1
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("UnionDataFrames").getOrCreate()
data1 = [("apple", 3, 5),
         ("banana", 1, 10),
         ("orange", 2, 8)]
df1 = spark.createDataFrame(data1, ["Name", "Col_1", "Col_2"])
data2 = [("apple", 3, 5),
         ("banana", 1, 15),
         ("grape", 4, 6)]
df2 = spark.createDataFrame(data2, ["Name", "Col_1", "Col_3"])
df2_renamed = df2.withColumnRenamed("Col_3", "Col_2")
df_union = df1.unionByName(df2_renamed)
df_union.show()


+------+-----+-----+
|  Name|Col_1|Col_2|
+------+-----+-----+
| apple|    3|    5|
|banana|    1|   10|
|orange|    2|    8|
| apple|    3|    5|
|banana|    1|   15|
| grape|    4|    6|
+------+-----+-----+



In [2]:
#3
from pyspark.sql.functions import first
spark = SparkSession.builder.appName("PivotExample").getOrCreate()
data = [
    (2021, 1, "US", 5000),
    (2021, 1, "EU", 4000),
    (2021, 2, "US", 5500),
    (2021, 2, "EU", 4500),
    (2021, 3, "US", 6000),
    (2021, 3, "EU", 5000),
    (2021, 4, "US", 7000),
    (2021, 4, "EU", 6000)
]

df = spark.createDataFrame(data, ["year", "quarter", "region", "revenue"])

# Pivot the DataFrame
pivot_df = (
    df.groupBy("year", "quarter")
      .pivot("region")                         # pivot on region column
      .agg(first("revenue"))                   # aggregate revenue values
      .orderBy("year", "quarter")
)

pivot_df.show()


+----+-------+----+----+
|year|quarter|  EU|  US|
+----+-------+----+----+
|2021|      1|4000|5000|
|2021|      2|4500|5500|
|2021|      3|5000|6000|
|2021|      4|6000|7000|
+----+-------+----+----+



In [3]:
#5
from pyspark.sql.functions import col, sum as _sum, when
spark = SparkSession.builder.appName("MissingValuesCheck").getOrCreate()
data = [
    ("A", 1, None),
    ("B", None, 123),
    ("B", 3, 456),
    ("D", None, None)
]
df = spark.createDataFrame(data, ["Name", "Value", "id"])

# Count missing (null) values for each column
missing_counts = df.select([
    _sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

# Check if there are any missing values at all
has_missing = any(v > 0 for v in missing_counts.values())
print(has_missing)
print(missing_counts)


True
{'Name': 0, 'Value': 2, 'id': 2}


In [4]:
#6
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
spark = SparkSession.builder.appName("NthRowFilter").getOrCreate()
data = [
    ("Alice", 1),
    ("Bob", 2),
    ("Charlie", 3),
    ("Dave", 4),
    ("Eve", 5),
    ("Frank", 6),
    ("Grace", 7),
    ("Hannah", 8),
    ("Igor", 9),
    ("Jack", 10)
]

df = spark.createDataFrame(data, ["Name", "Number"])

# Add row number
windowSpec = Window.orderBy("Number")
df_with_rn = df.withColumn("rn", row_number().over(windowSpec))

# Choose nth row,here every 5th row is chosen
n = 5
filtered_df = df_with_rn.filter(col("rn") % n == 0)

filtered_df.show()


+----+------+---+
|Name|Number| rn|
+----+------+---+
| Eve|     5|  5|
|Jack|    10| 10|
+----+------+---+



In [5]:
#7
from pyspark.sql.functions import col, when
spark = SparkSession.builder.appName("ColumnMatchCheck").getOrCreate()
data = [
    ("John", "John"),
    ("Lily", "Lucy"),
    ("Sam", "Sam"),
    ("Lucy", "Lily")
]

df = spark.createDataFrame(data, ["Name1", "Name2"])

# Compare columns and create Match column
df_result = df.withColumn("Match", when(col("Name1") == col("Name2"), True).otherwise(False))

df_result.show()


+-----+-----+-----+
|Name1|Name2|Match|
+-----+-----+-----+
| John| John| true|
| Lily| Lucy|false|
|  Sam|  Sam| true|
| Lucy| Lily|false|
+-----+-----+-----+



In [6]:
#8
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, lead, col
spark = SparkSession.builder.appName("LagLeadExample").getOrCreate()
data = [
    ("2023-01-01", "Store1", 100),
    ("2023-01-02", "Store1", 150),
    ("2023-01-03", "Store1", 200),
    ("2023-01-04", "Store1", 250),
    ("2023-01-05", "Store1", 300),
    ("2023-01-01", "Store2", 50),
    ("2023-01-02", "Store2", 60),
    ("2023-01-03", "Store2", 80),
    ("2023-01-04", "Store2", 90),
    ("2023-01-05", "Store2", 120)
]

df = spark.createDataFrame(data, ["Date", "Store", "Sales"])

# Define window by Store and ordered by Date
windowSpec = Window.partitionBy("Store").orderBy("Date")

df_with_lags = df.withColumn("Lag_Sales", lag("Sales", 1).over(windowSpec)) \
                 .withColumn("Lead_Sales", lead("Sales", 1).over(windowSpec))

df_with_lags.show()


+----------+------+-----+---------+----------+
|      Date| Store|Sales|Lag_Sales|Lead_Sales|
+----------+------+-----+---------+----------+
|2023-01-01|Store1|  100|     NULL|       150|
|2023-01-02|Store1|  150|      100|       200|
|2023-01-03|Store1|  200|      150|       250|
|2023-01-04|Store1|  250|      200|       300|
|2023-01-05|Store1|  300|      250|      NULL|
|2023-01-01|Store2|   50|     NULL|        60|
|2023-01-02|Store2|   60|       50|        80|
|2023-01-03|Store2|   80|       60|        90|
|2023-01-04|Store2|   90|       80|       120|
|2023-01-05|Store2|  120|       90|      NULL|
+----------+------+-----+---------+----------+



In [7]:
#9
from pyspark.sql.functions import col, explode, array
spark = SparkSession.builder.appName("ValueFrequency").getOrCreate()
data = [
    (1, 2, 3),
    (2, 3, 4),
    (1, 2, 3),
    (4, 5, 6),
    (2, 3, 4)
]
df = spark.createDataFrame(data, ["Column1", "Column2", "Column3"])

# Combine all columns into a single array, then explode to get one column--5x3 =15 rows after exploding
df_single = df.select(explode(array([col(c) for c in df.columns])).alias("single_column"))

# Count frequency of each unique value
result = df_single.groupBy("single_column").count().orderBy(col("count").desc())

result.show()


+-------------+-----+
|single_column|count|
+-------------+-----+
|            3|    4|
|            2|    4|
|            4|    3|
|            1|    2|
|            6|    1|
|            5|    1|
+-------------+-----+



In [8]:
#10
from pyspark.sql.functions import monotonically_increasing_id, col
spark = SparkSession.builder.appName("ReverseRows").getOrCreate()
data = [
    (1, 2, 3, 4),
    (2, 3, 4, 5),
    (3, 4, 5, 6),
    (4, 5, 6, 7)
]
df = spark.createDataFrame(data, ["col_1", "col_2", "col_3", "col_4"])

# Add a unique increasing ID column which increases with every row in a column
df_with_id = df.withColumn("id", monotonically_increasing_id())

# Sort by the ID in descending order to reverse rows and remove the extra column
df_reversed = df_with_id.orderBy(col("id").desc()).drop("id")

df_reversed.show()



+-----+-----+-----+-----+
|col_1|col_2|col_3|col_4|
+-----+-----+-----+-----+
|    4|    5|    6|    7|
|    3|    4|    5|    6|
|    2|    3|    4|    5|
|    1|    2|    3|    4|
+-----+-----+-----+-----+



In [9]:
#2
from pyspark.sql.functions import col
spark = SparkSession.builder.appName("FilterEmails").getOrCreate()
data = [
    ("buying books at amazom.com",),
    ("rameses@egypt.com",),
    ("matt@t.co",),
    ("narendra@modi.com",)
]

df = spark.createDataFrame(data, ["value"])

# Filter rows that look like email addresses using regex (regular expression)
email_df = df.filter(col("value").rlike(r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'))

# Show the result
email_df.show(truncate=False)


+-----------------+
|value            |
+-----------------+
|rameses@egypt.com|
|matt@t.co        |
|narendra@modi.com|
+-----------------+

